# Proyecto para el Análisis de Datos

---

Imaginemos que trabajamos para una plataforma global de videojuegos online con millones de jugadores activos.

Vamos a generar datos que ayuden a la hora de tomar decisiones críticas sobre balanceo del juego, detección de trampas e informes de rendimiento...


## El Dataset: Partidas simuladas de videojuegos

Vamos a trabajar con un dataset simulado de gran tamaño generado con el módulo `random` de NumPy:

- **Puntuaciones** — Puntos obtenidos por cada jugador al finalizar la partida. Una puntuación anómalamente alta puede indicar el uso de trampas o exploits.
- **Duración** — Minutos que duró cada partida. Permite detectar partidas inusualmente cortas o largas.
- **Códigos de resultado** — Estado final de la partida: 0 (Victoria), 1 (Derrota), 2 (Abandono/AFK), 3 (Error de conexión/dato perdido).


## Objetivos del Proyecto

Los objetivos principales son los siguientes:

1. Estadística fundamental
2. Detección de partidas sospechosas
3. Limpieza e imputación de datos
4. Ranking con `np.argsort`


---
## Objetivo 1: Estadística fundamental

NumPy ofrece funciones estadísticas clave:

- `np.mean()` — calcula el promedio
- `np.median()` — el valor central de los datos
- `np.std()` — muestra la dispersión respecto al promedio
- `np.percentile()` — valor por debajo del cual cae un porcentaje de los datos


Primero, necesitamos un dataset masivo para que la diferencia de velocidad sea evidente. Usaremos **1.000.000 de registros de partidas**.


In [1]:
import numpy as np

# Definir el tamaño de nuestro dataset
NUM_PARTIDAS = 1_000_000

# Generar el array de Puntuaciones con distribución normal
# Usamos semilla para reproducibilidad
np.random.seed(7)
media_punt = 4500.0
desv_std_punt = 1200.0
datos_puntuaciones = np.random.normal(media_punt, desv_std_punt, NUM_PARTIDAS)

print(f'Tamaño del array: {datos_puntuaciones.size} partidas')
print(f'Primeras 5 puntuaciones: {datos_puntuaciones[:5]}')

Tamaño del array: 1000000 partidas
Primeras 5 puntuaciones: [6528.63084456 3940.87515535 4539.38419641 4989.0195396  3553.29236565]


---
Generamos también datos de **duración de partida** para complementar el análisis:

In [2]:
# Generar el array de Duración (distribución uniforme entre 5 y 90 minutos)
np.random.seed(8)  # Semilla diferente
datos_duracion = np.random.uniform(5, 90, NUM_PARTIDAS)
print('Creado datos_duracion')

Creado datos_duracion


In [3]:
print('\n--- Estadísticas de Puntuaciones ---')

punt_media    = np.mean(datos_puntuaciones)
punt_mediana  = np.median(datos_puntuaciones)
punt_desv_std = np.std(datos_puntuaciones)

#calcular el percentil 10%
punt_p10      = np.percentile(datos_puntuaciones, 10)

#calcular el percentil 90%
punt_p90      = np.percentile(datos_puntuaciones, 90)

print(f'Media (Promedio):  {punt_media:.2f} puntos')
print(f'Mediana (Centro):  {punt_mediana:.2f} puntos')
print(f'Desv. Estándar:    {punt_desv_std:.2f} puntos')
print(f'Rango Normal 90%:  {punt_p10:.2f}  a  {punt_p90:.2f} puntos')


--- Estadísticas de Puntuaciones ---
Media (Promedio):  4499.47 puntos
Mediana (Centro):  4499.72 puntos
Desv. Estándar:    1201.02 puntos
Rango Normal 90%:  2959.63  a  6034.93 puntos


**Interpretación:**

- **Media ~X pts** — 
- **Mediana ~X pts** —
- **Desviación estándar X pts** — 
- **Percentiles** — El 80% está entre X y X 


---
## Objetivo 2: Detección de partidas sospechosas

Usaremos un array de códigos de resultado:
- **0** → Victoria
- **1** → Derrota
- **2** → Abandono
- **3** → Error de conexión

Objetivo: encontrar partidas con resultado **AFK o Abandono** y puntuación anómalamente alta (más de 2 desviaciones estándar sobre la media). Este patrón es una señal clásica de **uso de trampas o exploits**: el jugador acumula puntos de forma irregular y abandona antes de que la partida finalice oficialmente.


In [4]:
np.random.seed(7)

codigos_resultado = np.random.choice(
    a=[0, 1, 2, 3],
    size=NUM_PARTIDAS,
    p=[0.60, 0.28, 0.08, 0.04]  # 60% Victoria, 28% Derrota, 8% Abandono, 4% Error
)

print('Primeros 5 códigos de resultado:', codigos_resultado[:5])

Primeros 5 códigos de resultado: [0 1 0 1 3]


In [5]:
# Umbral de sospecha: media + 2 veces la desviación estándar
UMBRAL_SOSPECHA = punt_media + 2 * punt_desv_std
print(f'Umbral de Puntuación Sospechosa: {UMBRAL_SOSPECHA:.2f} puntos')

Umbral de Puntuación Sospechosa: 6901.50 puntos


In [6]:
# Máscara 1: partidas con resultado Abandono (código 2)
mascara_abandono = (codigos_resultado == 2)

# Máscara 2: partidas con puntuación superior al umbral
mascara_punt_alta = (datos_puntuaciones > UMBRAL_SOSPECHA)

print(f'Partidas con Abandono:          {mascara_abandono.sum()}')
print(f'Puntuaciones anómalamente altas: {mascara_punt_alta.sum()}')

Partidas con Abandono:          80093
Puntuaciones anómalamente altas: 22664


In [7]:
# Combinar ambas condiciones con AND lógico (&)
mascara_sospechosas = mascara_abandono & mascara_punt_alta

partidas_sospechosas = datos_puntuaciones[mascara_sospechosas]
cantidad_sospechosas = partidas_sospechosas.size

print('\n--- Resultados Finales ---')
print(f'Total de Partidas Sospechosas: {cantidad_sospechosas}')
print(f'Puntuaciones de muestra:       {partidas_sospechosas[:5]}')


--- Resultados Finales ---
Total de Partidas Sospechosas: 1853
Puntuaciones de muestra:       [7197.11406337 7131.09437445 7125.02091113 6939.97862677 7227.89398172]


De un millón de partidas, el sistema detectó **XXX partidas sospechosas** con puntuaciones entre XXX y XXX puntos, muy por encima del rango normal de XXX pts.

Esta combinación de condiciones (Abandono + puntuación extrema) es imposible de revisar manualmente en un millón de registros. Con enmascaramiento booleano lo resolvemos en milisegundos.


---
## Objetivo 3: Limpieza e Imputación de Datos

Los registros con código de resultado `3` (Error de conexión) no representan una partida real — el dato de puntuación es ruido. Deben reemplazarse con un valor estadísticamente válido para no sesgar el análisis posterior.

Usaremos la **mediana** de los datos válidos, ya que es robusta frente a valores extremos.

La función `np.where()` permite el reemplazo condicional:
```
np.where(Condicion, Valor_si_Verdadero, Valor_si_Falso)
```


In [9]:
# Máscara para datos válidos (resultado distinto de 3)
mascara_datos_validos =  (codigos_resultado != 3)

# Calcular la mediana solo sobre datos válidos
mediana_punt_validos = (np.median(mascara_datos_validos))
print(f'Mediana de partidas válidas: {mediana_punt_validos:.2f} puntos')

Mediana de partidas válidas: 1.00 puntos


In [10]:
# Reemplazar registros defectuosos con la mediana usando np.where()
datos_punt_limpios = np.where(
    mascara_datos_validos,   # CONDICIÓN: ¿error de conexión?
    mediana_punt_validos,     # SÍ: reemplazar con la mediana
    mediana_punt_validos        # NO: conservar valor original
)

cantidad_imputados = np.count_nonzero(~mascara_datos_validos)
print(f'Total de registros defectuosos imputados: {cantidad_imputados}')

Total de registros defectuosos imputados: 39880


In [11]:
# Verificar que la limpieza no altera la distribución
print(f'Mediana Original: {np.median(datos_puntuaciones):.2f} puntos')
print(f'Mediana Limpia:   {np.median(datos_punt_limpios):.2f} puntos')

Mediana Original: 4499.72 puntos
Mediana Limpia:   1.00 puntos


---
## Objetivo 4: Ranking con `np.argsort`

Una vez limpios los datos, queremos **ordenar y rankear** las partidas por puntuación. `np.argsort()` no devuelve los valores ordenados, sino los **índices** que los ordenarían — lo que nos permite operar sobre varios arrays a la vez manteniendo la correspondencia entre ellos.

```
np.argsort(array)  →  array de índices que ordenarían el array original de menor a mayor
```

Usaremos `np.argsort` para:
1. Obtener el **Top 10** de mejores partidas
2. Obtener el **Bottom 10** de peores partidas
3. Clasificar todas las partidas en tramos de rendimiento con `np.searchsorted`


In [12]:
# Obtener los índices que ordenarían el array de menor a mayor
indices_ordenados = np.argsort(datos_punt_limpios)

# Top 10: los últimos 10 índices (los de mayor puntuación), invertidos
top10_indices = indices_ordenados[-10:][::-1]
# Bottom 10: los primeros 10 índices (los de menor puntuación)
bottom10_indices = indices_ordenados[:10]

top10_punts = datos_punt_limpios[top10_indices]
bottom10_punts = datos_punt_limpios[bottom10_indices]

print('--- Top 10 Partidas (Mayor Puntuación) ---')
for i, idx in enumerate(top10_indices, 1):
	print(f'  {i:2d}. Índice {idx:7d}: {top10_punts[i-1]:8.2f} puntos')

--- Top 10 Partidas (Mayor Puntuación) ---
   1. Índice  999999:     1.00 puntos
   2. Índice  999998:     1.00 puntos
   3. Índice  999997:     1.00 puntos
   4. Índice  999996:     1.00 puntos
   5. Índice  999995:     1.00 puntos
   6. Índice  999994:     1.00 puntos
   7. Índice  999993:     1.00 puntos
   8. Índice  999992:     1.00 puntos
   9. Índice  999991:     1.00 puntos
  10. Índice  999990:     1.00 puntos


In [13]:
print('--- Bottom 10 Partidas (Menor Puntuación) ---')
# imprimir las partidas

--- Bottom 10 Partidas (Menor Puntuación) ---


In [14]:
# VAMOS A ANALIZAR LAS PARTIDAS POR TRAMOS

# Clasificar todas las partidas en tramos de rendimiento
# usando los percentiles 25, 50 y 75 como fronteras
p25 = np.percentile(datos_punt_limpios, 25)
p50 = np.percentile(datos_punt_limpios, 50)
p75 = np.percentile(datos_punt_limpios, 75)

fronteras = [p25, p50, p75]
tramos = np.searchsorted(fronteras, datos_punt_limpios)

nombres_tramos = ['Básico', 'Regular', 'Avanzado', 'Élite']
print('--- Distribución de Partidas por Tramo ---')
for i, nombre in enumerate(nombres_tramos):
    cantidad = (tramos == i).sum()
    porcentaje = cantidad / NUM_PARTIDAS * 100
    print(f'  {nombre:<10}: {cantidad:>7} partidas  ({porcentaje:.1f}%)')

--- Distribución de Partidas por Tramo ---
  Básico    : 1000000 partidas  (100.0%)
  Regular   :       0 partidas  (0.0%)
  Avanzado  :       0 partidas  (0.0%)
  Élite     :       0 partidas  (0.0%)


`np.argsort` es especialmente poderoso porque trabaja con **índices**, no con valores. Esto significa que podemos usar `top10_indices` para consultar simultáneamente la duración, el código de resultado o cualquier otro array que comparta la misma posición — sin necesidad de ordenar cada array por separado.

La clasificación por tramos con `np.searchsorted` es O(log n) por elemento — mucho más eficiente que comparaciones anidadas con bucles.
